# Retrieve BindingDB data

The purpose of this notebook is to retrieve affinity data based on local crystal structures from Binding DB.

In [7]:
import requests
import time
import csv

import pathlib
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.notebook import tqdm

Protein and data variables.

In [2]:
readout = 'affinity'

In [3]:
# define paths
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{readout}"
DATA.mkdir(parents=True, exist_ok=True)

### Download bioactivity data from BindingDB.

In [ ]:
root_dir = Path(DATA)

records = []
for uniprot_dir in root_dir.iterdir():
    if uniprot_dir.is_dir():
        uniprot_id = uniprot_dir.name
        for pdb_file in uniprot_dir.glob("*.pdb"):
            pdb_id = pdb_file.stem.upper()
            records.append({"uniprot_id": uniprot_id, "pdb_id": pdb_id, "pdb_path": str(pdb_file)})

df_pdbs = pd.DataFrame(records)
df_pdbs.head()


,uniprot_id,pdb_id,pdb_path
0,O08675,2PUX,/home/corey/Documents/comp_chem/ml/affinity/si...
1,Q99788,7YKD,/home/corey/Documents/comp_chem/ml/affinity/si...
2,Q99788,9L3Z,/home/corey/Documents/comp_chem/ml/affinity/si...
3,Q99788,9L3W,/home/corey/Documents/comp_chem/ml/affinity/si...
4,Q99788,8ZJG,/home/corey/Documents/comp_chem/ml/affinity/si...


In [24]:
import aiohttp
import asyncio
from tqdm.asyncio import tqdm_asyncio
from bs4 import BeautifulSoup

async def fetch_bindingdb_html(session, uniprot_id, retries=3, backoff=0.75):
    """
    Fetch and parse BindingDB affinities for a given UniProt ID from the HTML page.
    Returns (uniprot_id, list_of_entries or None)
    """
    url = f"https://www.bindingdb.org/bind/ByUniprotAccession.jsp?Uniprot_acc={uniprot_id}"

    for attempt in range(retries):
        try:
            async with session.get(url, timeout=20) as resp:
                html = await resp.text()
                if resp.status != 200 or "No data found" in html:
                    await asyncio.sleep(backoff * (2 ** attempt))
                    continue

                soup = BeautifulSoup(html, "html.parser")
                table = soup.find("table", {"class": "data"})
                if not table:
                    return uniprot_id, None

                rows = []
                headers = [th.text.strip() for th in table.find_all("th")]
                for row in table.find_all("tr")[1:]:
                    cols = [td.text.strip() for td in row.find_all("td")]
                    if not cols or len(cols) < 3:
                        continue
                    entry = dict(zip(headers[:len(cols)], cols))
                    entry["UniProt_ID"] = uniprot_id
                    rows.append(entry)

                return uniprot_id, rows if rows else None

        except Exception as e:
            await asyncio.sleep(backoff * (2 ** attempt))

    return uniprot_id, None


async def query_bindingdb_uniprot_html_async(uniprot_ids, concurrency=8, retries=3):
    """Run concurrent queries for a list of UniProt IDs."""
    connector = aiohttp.TCPConnector(limit=concurrency)
    timeout = aiohttp.ClientTimeout(total=30)
    results = []

    async with aiohttp.ClientSession(connector=connector, timeout=timeout) as session:
        tasks = [fetch_bindingdb_html(session, uid, retries=retries) for uid in uniprot_ids]
        for uniprot_id, entries in await tqdm_asyncio.gather(*tasks, desc="BindingDB HTML queries", unit="uniprot"):
            results.append((uniprot_id, entries))
    return results


In [25]:
uniprot_ids = df_pdbs["uniprot_id"].unique().tolist()

results = await query_bindingdb_uniprot_html_async(uniprot_ids, concurrency=10)


BindingDB HTML queries: 100%|██████████| 201/201 [00:15<00:00, 12.85uniprot/s]


In [26]:
import pandas as pd

rows = []
for uid, entries in results:
    if entries:
        for e in entries:
            e["UniProt_ID"] = uid
            rows.append(e)

bindingdb_affinity_df = pd.DataFrame(rows)
bindingdb_affinity_df.head()


""


In [14]:
df_flat = (
    df_pdbs.explode("bindingdb_entries")
    .reset_index(drop=True)
)

# Split dict columns into separate columns
bindingdb_df = pd.concat(
    [df_flat.drop(columns=["bindingdb_entries"]),
     df_flat["bindingdb_entries"].apply(pd.Series)],
    axis=1
)

bindingdb_df.to_csv("bindingdb_affinities_by_pdb.csv", index=False)
bindingdb_df.head()


,uniprot_id,pdb_id,pdb_path,bindingdb_raw,0
0,O08675,2PUX,/home/corey/Documents/comp_chem/ml/affinity/si...,None,NaN
1,Q99788,7YKD,/home/corey/Documents/comp_chem/ml/affinity/si...,None,NaN
2,Q99788,9L3Z,/home/corey/Documents/comp_chem/ml/affinity/si...,None,NaN
3,Q99788,9L3W,/home/corey/Documents/comp_chem/ml/affinity/si...,None,NaN
4,Q99788,8ZJG,/home/corey/Documents/comp_chem/ml/affinity/si...,None,NaN


In [34]:
# Dump to file.

filtered_df.to_csv(DATA / f"{readout}_chembl.csv")